In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from datetime import datetime, timezone, timedelta
import logging
import pickle
import math
import asyncio


logging.basicConfig(
    level=logging.INFO,  # Set the logging level
    format='%(asctime)s - %(levelname)s - %(message)s',  # Format for the log messages
    handlers=[
        logging.StreamHandler()  # Log to the console
    ]
)

%reload_ext autoreload
%autoreload 2
from data.raw.retrievers.alpaca_markets_retriever import AlpacaMarketsRetriever
from config.constants import *
from data.raw.retrievers.alpaca_portfolio_selection import (
    calculate_portfolio_cap_share,
    get_daily_stats,
    inspect_portfolio_history_depths,
    select_portfolio,
)
from data.raw.retrievers.stooq_utils import prepare_data_stooq
from data.processed.dataset_creation import DatasetCreator
from data.processed.indicators import *
from data.processed.targets import Balanced3ClassClassification
from data.processed.normalization import ZScoreOverWindowNormalizer, ZScoreNormalizer, MinMaxNormalizer
from data.processed.missing_values_handling import DummyMissingValuesHandler
from data.processed.dataset_pytorch import DatasetPytorch
from modeling.trainer import Trainer

from config.train_config import load_train_config
from config.constants import *

config = load_train_config()


In [2]:
from alpaca.data.timeframe import TimeFrame

retriever = AlpacaMarketsRetriever()
all_symbols = retriever.get_all_symbols()
len(all_symbols)

4880

In [4]:
portfolio, skipped_assets = await select_portfolio(
    all_symbols,
    start_date=config.data_config.end - timedelta(days=30),
    end_date=config.data_config.end,
    min_history_depth=datetime(2024, 10, 1, tzinfo=Constants.Data.EASTERN_TZ),
    portfolio_size=100,
    criteria='E_1m',
)
portfolio, skipped_assets

2026-08-10 19:07:06,019 - INFO - Starting performance sweep for 21 days...
2026-08-10 19:07:06,019 - INFO - Processing day 2026-07-02 00:00:00-04:00
2026-08-10 19:07:18,168 - INFO - Retrieving Alpaca bars from the Alpaca API for 2026-07-02 09:30:00-04:00 to 2026-07-02 16:00:00-04:00.
2026-08-10 19:07:18,381 - INFO - Retrieving Alpaca bars from the Alpaca API for 2026-07-02 09:30:00-04:00 to 2026-07-02 16:00:00-04:00.
2026-08-10 19:07:18,521 - INFO - Retrieving Alpaca bars from the Alpaca API for 2026-07-02 09:30:00-04:00 to 2026-07-02 16:00:00-04:00.
2026-08-10 19:07:18,527 - INFO - Retrieving Alpaca bars from the Alpaca API for 2026-07-02 09:30:00-04:00 to 2026-07-02 16:00:00-04:00.
2026-08-10 19:07:18,694 - INFO - Retrieving Alpaca bars from the Alpaca API for 2026-07-02 09:30:00-04:00 to 2026-07-02 16:00:00-04:00.
2026-08-10 19:07:19,162 - INFO - Retrieving Alpaca bars from the Alpaca API for 2026-07-02 09:30:00-04:00 to 2026-07-02 16:00:00-04:00.
2026-08-10 19:07:19,350 - INFO - Re

([('SPY', 4.266987025192916),
  ('QQQ', 4.213872054256445),
  ('NVDA', 3.4590069653796607),
  ('TQQQ', 3.2070934665685495),
  ('DRAM', 3.0975527526512243),
  ('AAPL', 2.7701924048283906),
  ('IWM', 2.718159224243684),
  ('INTC', 2.6910141278015582),
  ('GOOGL', 2.011432492361079),
  ('VOO', 1.8568282198582506),
  ('IREN', 1.8387141669866958),
  ('XLK', 1.8364173479478063),
  ('NFLX', 1.8189995029269956),
  ('IVV', 1.8096797091524286),
  ('AMZN', 1.7581276840007667),
  ('MU', 1.7464550977957376),
  ('EWY', 1.7459333841123972),
  ('GDX', 1.727430115882174),
  ('TSLA', 1.7106506485734816),
  ('SMH', 1.7002798708039188),
  ('XLV', 1.6823134847291157),
  ('RSP', 1.6374090221545026),
  ('MSFT', 1.6171444327877584),
  ('PLTR', 1.582550024358364),
  ('SOXX', 1.5686125029066025),
  ('WMT', 1.517766922600632),
  ('KLAC', 1.50806496787276),
  ('XLY', 1.5047731697480948),
  ('QQQM', 1.4704060864492865),
  ('DIA', 1.4582805902200164),
  ('WULF', 1.4549553807965891),
  ('VTI', 1.452683589353541),
  

In [9]:
porfolio_sorted = sorted([asset for asset, score in portfolio])
porfolio_sorted

['AAPL',
 'AMD',
 'AMZN',
 'APLD',
 'ARKK',
 'ASX',
 'AVGO',
 'BAC',
 'BKNG',
 'BKR',
 'BMY',
 'BSX',
 'CIFR',
 'CSCO',
 'CSX',
 'CTSH',
 'DIA',
 'DRAM',
 'DVN',
 'EEM',
 'EFA',
 'EWJ',
 'EWT',
 'EWY',
 'FCX',
 'GDX',
 'GLD',
 'GOOG',
 'GOOGL',
 'HOOD',
 'HPE',
 'IEFA',
 'IEMG',
 'IGV',
 'IJR',
 'INTC',
 'IONQ',
 'IREN',
 'IVV',
 'IVW',
 'IWF',
 'IWM',
 'IYR',
 'KLAC',
 'KO',
 'KRE',
 'MARA',
 'MRVL',
 'MSFT',
 'MSTR',
 'MU',
 'NBIS',
 'NEE',
 'NFLX',
 'NKE',
 'NOW',
 'NVDA',
 'ORCL',
 'OXY',
 'PLTR',
 'PYPL',
 'QBTS',
 'QLD',
 'QQQ',
 'QQQM',
 'RIOT',
 'RKLB',
 'RSP',
 'SLB',
 'SLV',
 'SMCI',
 'SMH',
 'SOXQ',
 'SOXX',
 'SPXL',
 'SPY',
 'SPYG',
 'STM',
 'TFC',
 'TQQQ',
 'TSLA',
 'TSM',
 'USB',
 'VGT',
 'VNQ',
 'VOO',
 'VTI',
 'VTWO',
 'VUG',
 'VZ',
 'WMT',
 'WULF',
 'XLC',
 'XLE',
 'XLI',
 'XLK',
 'XLP',
 'XLV',
 'XLY',
 'XOM']

In [4]:
inspect_portfolio_history_depths(portfolio, retriever)

SPY 2016-01-01 00:01:00+00:00
QQQ 2016-01-01 00:00:00+00:00
NVDA 2016-01-04 11:37:00+00:00
TQQQ 2016-01-01 00:06:00+00:00
DRAM 2016-01-04 16:38:00+00:00
AAPL 2016-01-01 00:00:00+00:00
IWM 2016-01-01 00:11:00+00:00
INTC 2016-01-01 00:48:00+00:00
GOOGL 2016-01-04 09:00:00+00:00
VOO 2016-01-04 13:00:00+00:00
IREN 2021-11-17 16:44:00+00:00
XLK 2016-01-04 11:57:00+00:00
NFLX 2016-01-01 00:03:00+00:00
IVV 2016-01-04 12:02:00+00:00
AMZN 2016-01-01 00:56:00+00:00
MU 2016-01-01 00:12:00+00:00
EWY 2016-01-04 14:28:00+00:00
GDX 2016-01-01 00:15:00+00:00
TSLA 2016-01-01 00:30:00+00:00
SMH 2016-01-04 14:30:00+00:00
XLV 2016-01-04 13:00:00+00:00
RSP 2016-01-04 14:30:00+00:00
MSFT 2016-01-01 00:02:00+00:00
PLTR 2020-09-30 17:38:00+00:00
SOXX 2016-01-04 13:46:00+00:00
WMT 2016-01-04 09:55:00+00:00
KLAC 2016-01-04 14:30:00+00:00
XLY 2016-01-04 13:44:00+00:00
QQQM 2020-10-13 13:38:00+00:00
DIA 2016-01-01 00:08:00+00:00
WULF 2016-01-05 15:37:00+00:00
VTI 2016-01-04 13:00:00+00:00
SNDK 2025-02-13 14:42:00

['SPY',
 'QQQ',
 'NVDA',
 'TQQQ',
 'DRAM',
 'AAPL',
 'IWM',
 'INTC',
 'GOOGL',
 'VOO',
 'IREN',
 'XLK',
 'NFLX',
 'IVV',
 'AMZN',
 'MU',
 'EWY',
 'GDX',
 'TSLA',
 'SMH',
 'XLV',
 'RSP',
 'MSFT',
 'PLTR',
 'SOXX',
 'WMT',
 'KLAC',
 'XLY',
 'QQQM',
 'DIA',
 'WULF',
 'VTI',
 'SNDK',
 'GLD',
 'BKNG',
 'EWT',
 'IWF',
 'SMCI',
 'KO',
 'ASX',
 'KRE',
 'HOOD',
 'CSCO',
 'GOOG',
 'NOW',
 'IONQ',
 'VGT',
 'QLD',
 'CRWV',
 'CIFR',
 'XLI',
 'SLV',
 'XLC',
 'STM',
 'IYR',
 'ARKK',
 'IGV',
 'VUG',
 'XLE',
 'BMY',
 'BAC',
 'EEM',
 'APLD',
 'IJR',
 'IVW',
 'TSM',
 'IEMG',
 'VZ',
 'AVGO',
 'XLP',
 'EFA',
 'IEFA',
 'VNQ',
 'VTWO',
 'XOM',
 'BKR',
 'BSX',
 'RKLB',
 'RIOT',
 'FIG',
 'QBTS',
 'EWJ',
 'SPYG',
 'CSX',
 'BMNR',
 'NKE',
 'MARA',
 'NEE',
 'MRVL',
 'AMD',
 'PYPL',
 'FCX',
 'MSTR',
 'CTSH',
 'SOXQ',
 'SLB',
 'HPE',
 'ORCL',
 'NBIS',
 'DVN']

In [5]:
await calculate_portfolio_cap_share(portfolio, config.data_config.end)

ZeroDivisionError: division by zero